# Crime x preço de imóvel (ITBI)

Segunda análise, independente do Airbnb: relaciona o crime mensal por CISP com o preço por m² de apartamentos e casas vendidos no Rio, a partir da tabela de ITBI da Prefeitura (2014 em diante), com muito mais meses e transações que os 3 snapshots do Airbnb.

In [ ]:
from pathlib import Path
import geopandas as gpd
import pandas as pd

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "data").is_dir()
)
DATA_DIR = PROJECT_ROOT / "data"

In [ ]:
crimes = pd.read_csv("../../data/crime_data/BaseDPEvolucaoMensalCisp.csv", encoding="latin-1", sep=";")
crimes

In [ ]:
cisp = gpd.read_file(
    "../../data/shape_file/CISPshp/CISPshp/cisp.shp"
)

aisp = gpd.read_file(
    "../../data/shape_file/AISPshp/SHP/aisp.shp"
)

risp = gpd.read_file(
    "../../data/shape_file/RISPshp/RISPshp/risp.shp"
)

## População por CISP

A população de cada CISP vem do Censo 2022, somando os setores censitários, como no EDA_Per_Capita.

In [ ]:
import zipfile

POP_ZIP = DATA_DIR / "populacao" / "População_Residente_e_Domicílios__por_setor_censitário_-_Censo_2022.zip"
SETORES_ZIP = DATA_DIR / "populacao" / "RJ_setores_CD2022.zip"  # malha de setores do IBGE (Censo 2022)
MUNICIPIO_RIO = "3304557"

# População por setor censitário (Data.Rio / Censo 2022).
with zipfile.ZipFile(POP_ZIP) as z:
    csv_name = next(n for n in z.namelist() if n.endswith(".csv"))
    pop_setores = pd.read_csv(
        z.open(csv_name), sep=";", encoding="latin-1", usecols=["CD_SETOR", "Total de pessoas"]
    ).rename(columns={"Total de pessoas": "pessoas"})
assert pop_setores["CD_SETOR"].is_unique

# Polígonos dos setores da capital, com a população.
setores = gpd.read_file(f"zip://{SETORES_ZIP.as_posix()}", where=f"CD_MUN='{MUNICIPIO_RIO}'")
setores["CD_SETOR"] = setores["CD_SETOR"].astype("int64")
setores = setores[["CD_SETOR", "geometry"]].merge(pop_setores, on="CD_SETOR", how="inner", validate="one_to_one")
assert len(setores) == len(pop_setores), "setor do CSV sem polígono na malha"

# Cada setor vai para a CISP que contém o seu ponto representativo (calculado em CRS métrico).
cisp_poligonos = cisp[["cisp", "geometry"]].copy()
cisp_poligonos["cisp"] = pd.to_numeric(cisp_poligonos["cisp"]).astype(int)
pontos_setor = setores.copy()
pontos_setor["geometry"] = setores.to_crs(31983).representative_point().to_crs(cisp_poligonos.crs)
setor_cisp = gpd.sjoin(pontos_setor, cisp_poligonos, how="left", predicate="within")
assert setor_cisp["CD_SETOR"].is_unique, "ponto caiu em mais de uma CISP"

populacao_cisp = (
    setor_cisp.dropna(subset=["cisp"])
    .groupby("cisp", as_index=False)
    .agg(populacao=("pessoas", "sum"), setores=("CD_SETOR", "size"))
)
populacao_cisp["cisp"] = populacao_cisp["cisp"].astype("Int64")
area_cisp = cisp[["cisp", "areakm2"]].assign(cisp=lambda d: pd.to_numeric(d["cisp"]).astype("Int64"))
populacao_cisp = populacao_cisp.merge(area_cisp, on="cisp", how="left", validate="one_to_one")
populacao_cisp["hab_por_km2"] = populacao_cisp["populacao"] / populacao_cisp["areakm2"]

print(f"População da capital: {setores['pessoas'].sum():,} em {len(setores):,} setores")
print(f"Setores sem CISP: {setor_cisp['cisp'].isna().sum()} (população {setor_cisp.loc[setor_cisp['cisp'].isna(), 'pessoas'].sum():,})")
print(f"CISPs com população: {len(populacao_cisp)}")
display(populacao_cisp.sort_values("hab_por_km2").round(0).iloc[[0, 1, 2, -3, -2, -1]])

In [ ]:
import json
import time
import urllib.parse
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
from shapely import wkt

INICIO = "2014-01"     # depois da criação das CISPs 11 e 45 (2013): limites estáveis
MIN_TRANSACOES = 10    # transações mínimas por CISP x mês x tipologia
JANELA_CRIME = 3       # crime = média dos últimos 3 meses...
LAG_MESES = 1          # ...terminando 1 mês antes do preço
QUEBRA = "2022-01"     # STJ Tema 1.113: a base do ITBI passa a ser o valor declarado
MIN_SHARE_RUA = 0.7    # sensibilidade: rua com >= 70% do comprimento na CISP dominante
N_BOOT_WCB = 1999
RANDOM_STATE = 0

# Níveis de gravidade, do mais grave (4) ao menos grave (1), sem colunas sobrepostas (mesmos do EDA_Vini).
TIER_NAMES = {
    4: "Contra a vida / sexual",
    3: "Roubos",
    2: "Violência interpessoal",
    1: "Patrimonial sem violência",
}
TIER_COLUMNS = {
    4: ["letalidade_violenta", "tentat_hom", "feminicidio", "tentativa_feminicidio",
        "estupro", "sequestro", "sequestro_relampago", "extorsao"],
    3: ["total_roubos"],
    2: ["lesao_corp_dolosa", "ameaca"],
    1: ["total_furtos", "estelionato"],
}
WEIGHTS = {4: 4, 3: 3, 2: 2, 1: 1}

## Dados do ITBI e geometria dos logradouros

A tabela do ITBI traz transações por logradouro e mês, sem coordenadas. A camada de trechos de logradouros da Prefeitura tem o mesmo código `cl` e a geometria de cada rua. Os dois são baixados uma vez e guardados em `data/itbi/`.

In [ ]:
ITBI_DIR = DATA_DIR / "itbi"
ITBI_DIR.mkdir(exist_ok=True)
BASE_URL = "https://pgeo3.rio.rj.gov.br/arcgis/rest/services"
ITBI_URL = f"{BASE_URL}/Fazenda/ITBI/MapServer/8"
TRECHOS_URL = f"{BASE_URL}/CadLog/Trechos_Logradouros/MapServer/0"


def paginar(url, campos, geometria=False, tentativas=3):
    """Percorre uma camada do ArcGIS de 2000 em 2000 registros, na ordem do objectid."""
    deslocamento = 0
    while True:
        params = {"where": "1=1", "outFields": campos, "returnGeometry": str(geometria).lower(),
                  "resultOffset": deslocamento, "resultRecordCount": 2000, "orderByFields": "objectid", "f": "json"}
        if geometria:
            params["outSR"] = 4326
        for tentativa in range(tentativas):
            try:
                pagina = json.load(urllib.request.urlopen(f"{url}/query?{urllib.parse.urlencode(params)}", timeout=120))
                break
            except Exception:
                if tentativa == tentativas - 1:
                    raise
                time.sleep(2)
        if not pagina["features"]:
            return
        yield from pagina["features"]
        deslocamento += len(pagina["features"])


def linha_wkt(geometria):
    return "MULTILINESTRING (" + ", ".join("(" + ", ".join(f"{x} {y}" for x, y in p) + ")" for p in geometria["paths"]) + ")"


# Os dois downloads ficam em cache em data/itbi/ (a pasta data/ não é versionada).
if not (ITBI_DIR / "itbi.csv").exists():
    pd.DataFrame([f["attributes"] for f in paginar(ITBI_URL, "*")]).to_csv(ITBI_DIR / "itbi.csv", index=False, encoding="utf-8")
if not (ITBI_DIR / "trechos.csv").exists():
    pd.DataFrame(
        [{"cl": f["attributes"]["cl"], "wkt": linha_wkt(f["geometry"])} for f in paginar(TRECHOS_URL, "cl", geometria=True)]
    ).to_csv(ITBI_DIR / "trechos.csv", index=False, encoding="utf-8")

itbi_bruto = pd.read_csv(ITBI_DIR / "itbi.csv", dtype={"cl": str, "codbairro": str, "cd_utilizacao": str})
trechos = pd.read_csv(ITBI_DIR / "trechos.csv", dtype={"cl": str})
print(f"ITBI: {len(itbi_bruto):,} linhas rua x mês, {itbi_bruto['total_transações'].sum():,} transações, "
      f"{itbi_bruto['ano_transação'].min()} a {itbi_bruto['ano_transação'].max()}")
print(f"Trechos de logradouros: {len(trechos):,} em {trechos['cl'].nunique():,} logradouros")

## Logradouro → CISP

Cada rua vai para a CISP que contém a maior parte do seu comprimento. Ruas que cruzam mais de uma CISP ficam com a dominante, e há uma checagem de sensibilidade para elas mais abaixo.

In [ ]:
# Cada trecho vai para a CISP que contém o seu ponto médio (calculado em CRS métrico).
trechos_g = gpd.GeoDataFrame(trechos[["cl"]], geometry=trechos["wkt"].map(wkt.loads), crs=4326).to_crs(31983)
trechos_g["comprimento"] = trechos_g.length
meio = trechos_g.copy()
meio["geometry"] = trechos_g.interpolate(0.5, normalized=True)
trecho_cisp = gpd.sjoin(meio.to_crs(cisp_poligonos.crs), cisp_poligonos, how="left", predicate="within")
trecho_cisp = trecho_cisp[~trecho_cisp.index.duplicated(keep="first")]  # polígonos sobrepostos

# Cada logradouro (cl) fica com a CISP que tem a maior parte do seu comprimento.
por_rua = trecho_cisp.dropna(subset=["cisp"]).groupby(["cl", "cisp"], as_index=False)["comprimento"].sum()
por_rua["share"] = por_rua["comprimento"] / por_rua.groupby("cl")["comprimento"].transform("sum")
rua_cisp = (
    por_rua.sort_values(["cl", "share"], ascending=[True, False]).drop_duplicates("cl")[["cl", "cisp", "share"]]
)
rua_cisp["cisp"] = rua_cisp["cisp"].astype(int)

casou = itbi_bruto["cl"].isin(rua_cisp["cl"])
cobertura_rua = itbi_bruto.loc[casou, "total_transações"].sum() / itbi_bruto["total_transações"].sum()
assert cobertura_rua >= 0.95, "poucas transações com CISP"
print(f"Transações em logradouros com CISP: {cobertura_rua:.1%}")
print(f"Logradouros com menos de {MIN_SHARE_RUA:.0%} do comprimento na CISP dominante: {(rua_cisp['share'] < MIN_SHARE_RUA).mean():.1%}")

## Limpeza do ITBI

Ficam só imóveis residenciais (apartamento ou casa), de compra e venda e transferidos por inteiro. O preço é o valor do imóvel por m² de área construída, sem os extremos (p1 e p99) de cada ano e tipologia.

In [ ]:
itbi = itbi_bruto.rename(columns={
    "total_transações": "n", "média_valor_imóvel": "valor_imovel", "média_área_construída": "area",
    "média_percentual_transferido": "pct", "principais_tipologias": "tipologia",
    "ano_transação": "ano", "mês_transação": "mes", "principal_transação_mercado": "mercado",
}).merge(rua_cisp, on="cl", how="inner", validate="many_to_one")
for coluna in ("uso", "tipologia", "mercado"):
    itbi[coluna] = itbi[coluna].str.strip()

linhas_antes = len(itbi)
itbi = itbi[
    (itbi["uso"] == "RESIDENCIAL")
    & itbi["tipologia"].isin(["APARTAMENTO", "CASA"])
    & (itbi["mercado"] == "COMPRA E VENDA")
    & (itbi["pct"] >= 99.99)  # imóvel inteiro (transferência parcial distorce o valor)
    & (itbi["area"] > 0)
    & (itbi["valor_imovel"] > 0)
].copy()
itbi["data"] = pd.to_datetime(itbi["ano"].astype(str) + "-" + itbi["mes"].astype(str).str.zfill(2) + "-01")
itbi["preco_m2"] = itbi["valor_imovel"] / itbi["area"]

# Sem os extremos (p1 e p99) de cada ano e tipologia.
grupo = itbi.groupby(["ano", "tipologia"])["preco_m2"]
itbi = itbi[(itbi["preco_m2"] >= grupo.transform(lambda s: s.quantile(0.01))) & (itbi["preco_m2"] <= grupo.transform(lambda s: s.quantile(0.99)))]

# O último mês da tabela pode vir incompleto: tira os meses finais com menos da metade do volume habitual.
mensal = itbi.groupby("data")["n"].sum()
FIM = mensal.index.max()
while mensal[FIM] < 0.5 * mensal[:FIM].iloc[-13:-1].median():
    FIM -= pd.DateOffset(months=1)
itbi = itbi[(itbi["data"] >= INICIO) & (itbi["data"] <= FIM)]

print(f"Linhas: {linhas_antes:,} com CISP -> {len(itbi):,} após os filtros, de {itbi['data'].min():%m/%Y} a {itbi['data'].max():%m/%Y}")
print(itbi.groupby("tipologia").agg(linhas=("n", "size"), transacoes=("n", "sum"), preco_m2_mediano=("preco_m2", "median")).round(0))

## Preço por CISP, mês e tipologia

O preço de cada célula é ponderado pelo número de transações, e células com poucas transações são descartadas.

In [ ]:
def agregar_preco(base, min_n=MIN_TRANSACOES, freq="M"):
    """Uma linha por CISP x período x tipologia. Preço = Σ(valor x n) / Σ(área x n), ou seja, ponderado pelas transações."""
    d = base.assign(valor_n=base["valor_imovel"] * base["n"], area_n=base["area"] * base["n"])
    d["periodo"] = d["data"] if freq == "M" else d["data"].dt.to_period("Q").dt.start_time
    a = d.groupby(["cisp", "periodo", "tipologia"], as_index=False).agg(
        valor=("valor_n", "sum"), area=("area_n", "sum"), n=("n", "sum"), ruas=("cl", "nunique")
    )
    a["preco_m2"] = a["valor"] / a["area"]
    a["y"] = np.log(a["preco_m2"])
    return a[a["n"] >= (min_n if freq == "M" else 3 * min_n)].reset_index(drop=True)


preco = agregar_preco(itbi)
assert not preco.duplicated(["cisp", "periodo", "tipologia"]).any()
print(f"{len(preco):,} células CISP x mês x tipologia com pelo menos {MIN_TRANSACOES} transações, {preco['cisp'].nunique()} CISPs")
print("Transações por célula:", preco["n"].describe()[["min", "50%", "mean", "max"]].round(0).to_dict())

## Crime mensal por CISP

O crime vira taxa por 100 mil habitantes em quatro níveis de gravidade. A exposição de cada mês é a média dos últimos 3 meses, terminando um mês antes do preço, para o crime vir antes.

In [ ]:
# Crime mensal por CISP da capital, só CISPs que existem no shapefile (sai a CISP 45), a partir de INICIO.
crimes_rio = crimes.assign(cisp=pd.to_numeric(crimes["cisp"])).query("munic == 'Rio de Janeiro'")
crimes_rio = crimes_rio[crimes_rio["cisp"].isin(cisp_poligonos["cisp"])].copy()
crimes_rio["cisp"] = crimes_rio["cisp"].astype(int)
crimes_rio["data"] = pd.to_datetime(crimes_rio["ano"].astype(str) + "-" + crimes_rio["mes"].astype(str).str.zfill(2) + "-01")
crimes_rio = crimes_rio[crimes_rio["data"] >= INICIO]
assert not crimes_rio.duplicated(["cisp", "data"]).any()
for nivel, colunas in TIER_COLUMNS.items():
    crimes_rio[f"crimes_nivel{nivel}"] = crimes_rio[colunas].sum(axis=1)

grade = pd.MultiIndex.from_product(
    [sorted(crimes_rio["cisp"].unique()), pd.date_range(INICIO, crimes_rio["data"].max(), freq="MS")], names=["cisp", "data"]
)
cm = crimes_rio.set_index(["cisp", "data"]).reindex(grade)[[f"crimes_nivel{n}" for n in TIER_COLUMNS]]
info_cisp = populacao_cisp.assign(cisp=populacao_cisp["cisp"].astype(int)).set_index("cisp")[["populacao", "areakm2"]]
cm = cm.join(info_cisp, on="cisp")
print(f"Crime: {cm.index.get_level_values('cisp').nunique()} CISPs, {INICIO} a {crimes_rio['data'].max():%Y-%m}; CISPs sem população: {cm['populacao'].isna().groupby(level='cisp').any().sum()}")


def exposicao(cm, denominador="habitantes", lag=LAG_MESES):
    """Intensidade de crime: média ponderada (4-3-2-1) do log da taxa de cada nível. A taxa é a média dos últimos
    JANELA_CRIME meses, terminando `lag` meses antes, por 100 mil habitantes (ou por km²)."""
    base = cm["populacao"] / 100_000 if denominador == "habitantes" else cm["areakm2"]
    saida = pd.DataFrame(index=cm.index)
    for n in TIER_COLUMNS:
        media = cm.groupby(level="cisp")[f"crimes_nivel{n}"].transform(lambda s: s.rolling(JANELA_CRIME, min_periods=JANELA_CRIME).mean())
        taxa = (media / base).groupby(level="cisp").shift(lag)
        saida[f"log_taxa_nivel{n}"] = np.log(taxa.clip(lower=1e-3))
    saida["intensidade"] = sum(WEIGHTS[n] * saida[f"log_taxa_nivel{n}"] for n in WEIGHTS) / sum(WEIGHTS.values())
    return saida


expo_hab = exposicao(cm, "habitantes")
expo_km2 = exposicao(cm, "km2")
expo_hab0 = exposicao(cm, "habitantes", lag=0)

## Painel

Junta preço e crime por CISP e mês. O desvio-padrão do crime dentro da CISP mostra quanto poder o painel tem em comparação ao Airbnb.

In [ ]:
def montar_painel(min_n=MIN_TRANSACOES, tipologias=("APARTAMENTO", "CASA"), share_min=0.0, freq="M", expo=expo_hab, coluna_x="intensidade"):
    """Junta preço (CISP x período x tipologia) com a exposição ao crime da CISP no mesmo período."""
    base = itbi[itbi["tipologia"].isin(tipologias) & (itbi["share"] >= share_min)]
    p = agregar_preco(base, min_n, freq)
    e = expo.reset_index()
    e["periodo"] = e["data"] if freq == "M" else e["data"].dt.to_period("Q").dt.start_time
    e = e.groupby(["cisp", "periodo"], as_index=False)[[c for c in e.columns if c not in ("cisp", "data", "periodo")]].mean()
    p = p.merge(e, on=["cisp", "periodo"], how="inner", validate="many_to_one")
    p["intensidade"] = p[coluna_x]
    p = p.dropna(subset=["intensidade", "y"]).copy()
    p["tempo"] = p["periodo"].dt.strftime("%Y-%m")
    p["cisp_tip"] = p["cisp"].astype(str) + "_" + p["tipologia"]
    p["tempo_tip"] = p["tempo"] + "_" + p["tipologia"]
    return p.reset_index(drop=True)


painel = montar_painel()
assert not painel.duplicated(["cisp", "periodo", "tipologia"]).any()
assert painel[["y", "intensidade", "n"]].notna().all().all()
dp_total = painel["intensidade"].std()
dp_dentro = (painel["intensidade"] - painel.groupby("cisp_tip")["intensidade"].transform("mean")).std()
print(f"Painel: {len(painel):,} células, {painel['cisp'].nunique()} CISPs, {painel['periodo'].min():%m/%Y} a {painel['periodo'].max():%m/%Y}")
print(f"Desvio-padrão da intensidade de crime: {dp_total:.2f} no total, {dp_dentro:.2f} dentro de cada CISP x tipologia ao longo do tempo (no Airbnb: 0,67 e 0,15)")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
anual = painel.assign(ano=painel["periodo"].dt.year).groupby(["ano", "tipologia"]).apply(
    lambda g: np.average(g["preco_m2"], weights=g["n"]), include_groups=False
).unstack()
anual.plot(ax=axes[0], marker="o")
axes[0].set_title("R$/m² médio por ano, ponderado por transações", loc="left")
axes[0].set_ylabel("R$/m²")
painel.groupby("periodo")["cisp"].nunique().plot(ax=axes[1])
axes[1].set_title("CISPs com célula válida por mês", loc="left")
plt.tight_layout()
plt.show()

## Regressão

Regressão de ln(R$/m²) sobre a intensidade de crime, ponderada pelas transações, com efeitos fixos. O efeito fixo de mês absorve a inflação e o de CISP × tipologia compara cada CISP com ela mesma ao longo do tempo. Uma última especificação tira também a tendência linear de cada CISP, para separar as oscilações de curto prazo das mudanças lentas do bairro. O erro-padrão é agrupado por CISP (wild cluster bootstrap), e o período é dividido em 2022, quando o STJ mudou a base do ITBI.

In [ ]:
def residualizar(frame, efeitos, x="intensidade", peso="n", tendencias=()):
    """Teorema de Frisch-Waugh-Lovell com pesos: resíduos (ponderados) de y e de x depois de tirar os efeitos fixos
    e, se pedido, uma tendência linear no tempo própria de cada grupo em `tendencias`."""
    sw = np.sqrt(frame[peso].to_numpy(float))
    dummies = [pd.get_dummies(frame[e], drop_first=True, dtype=float).to_numpy() for e in efeitos]
    anos = ((frame["periodo"] - frame["periodo"].min()).dt.days / 365.25).to_numpy()
    dummies += [pd.get_dummies(frame[g], dtype=float).to_numpy() * anos[:, None] for g in tendencias]
    D = np.column_stack([np.ones(len(frame))] + dummies) * sw[:, None]
    alvo = np.column_stack([frame["y"].to_numpy(), frame[x].to_numpy()]) * sw[:, None]
    residuo = alvo - D @ np.linalg.lstsq(D, alvo, rcond=None)[0]
    return residuo[:, 0], residuo[:, 1]


def estimar(frame, efeitos, x="intensidade", n_boot=N_BOOT_WCB, tendencias=()):
    """β de ln(R$/m²) em x, com EP robusto a cluster (CISP), p e IC por wild cluster bootstrap (inversão do teste)."""
    yt, xt = residualizar(frame, efeitos, x, tendencias=tendencias)
    codigos, _ = pd.factorize(frame["cisp"])
    G = codigos.max() + 1
    sxy = np.bincount(codigos, weights=xt * yt)
    sxx_g = np.bincount(codigos, weights=xt * xt)
    sxx = sxx_g.sum()
    beta = sxy.sum() / sxx
    escores = sxy - beta * sxx_g
    ep = np.sqrt(G / (G - 1) * (escores @ escores)) / sxx
    W = np.random.default_rng(RANDOM_STATE).choice([-1.0, 1.0], size=(n_boot, G))

    def p_wcb(beta0):
        s0 = sxy - beta0 * sxx_g
        delta = W @ s0 / sxx
        escores_b = W * s0 - delta[:, None] * sxx_g
        ep_b = np.sqrt(G / (G - 1) * (escores_b ** 2).sum(axis=1)) / sxx
        return np.mean(np.abs(delta / ep_b) >= abs((beta - beta0) / ep))

    grade = beta + ep * np.linspace(-10, 10, 201)
    aceitos = grade[[p_wcb(b0) > 0.05 for b0 in grade]]
    ic_inf, ic_sup = aceitos.min(), aceitos.max()
    dp_t = frame[x].std()
    dp_d = (frame[x] - frame.groupby("cisp_tip")[x].transform("mean")).std()
    pct = lambda b, dp: (np.exp(b * dp) - 1) * 100  # variação % do preço por +1 desvio-padrão do crime
    return {"beta": beta, "ep": ep, "ic95_inf": ic_inf, "ic95_sup": ic_sup, "p_wcb": p_wcb(0.0),
            "efeito_pct_dp_total": pct(beta, dp_t), "pct_inf": pct(ic_inf, dp_t), "pct_sup": pct(ic_sup, dp_t),
            "efeito_pct_dp_dentro": pct(beta, dp_d), "mde_pct_dp_total": pct(2.8 * ep, dp_t),
            "clusters": int(G), "n": len(frame)}


ESPECIFICACOES = {  # (efeitos fixos, tendências lineares por grupo)
    "Só tempo e tipologia (entre CISPs)": (["tempo", "tipologia"], ()),
    "+ efeito fixo de CISP x tipologia (dentro da CISP)": (["tempo", "cisp_tip"], ()),
    "+ tempo próprio por tipologia": (["tempo_tip", "cisp_tip"], ()),
    "+ tendência linear própria de cada CISP x tipologia": (["tempo", "cisp_tip"], ("cisp_tip",)),
}
COLUNAS = ["beta", "ep", "p_wcb", "efeito_pct_dp_total", "pct_inf", "pct_sup", "efeito_pct_dp_dentro", "mde_pct_dp_total", "n"]

linhas = []
quebra = pd.Timestamp(QUEBRA)
for periodo_nome, recorte in [("2014 em diante", painel), (f"antes de {QUEBRA[:4]}", painel[painel["periodo"] < quebra]), (f"{QUEBRA[:4]} em diante", painel[painel["periodo"] >= quebra])]:
    for nome, (efeitos, tendencias) in ESPECIFICACOES.items():
        linhas.append({"periodo": periodo_nome, "especificacao": nome, **estimar(recorte, efeitos, tendencias=tendencias)})
principal = pd.DataFrame(linhas).set_index(["periodo", "especificacao"])
display(principal[COLUNAS].round(3))

## Robustez

A mesma regressão com outras escolhas de amostra e de medida de crime, e cada nível de gravidade separado.

In [ ]:
ESPEC_DENTRO = ["tempo", "cisp_tip"]
variantes = {
    "Principal": painel,
    "Só apartamentos": montar_painel(tipologias=("APARTAMENTO",)),
    "Mínimo de 5 transações": montar_painel(min_n=5),
    "Mínimo de 20 transações": montar_painel(min_n=20),
    "Ruas com >= 70% na CISP": montar_painel(share_min=MIN_SHARE_RUA),
    "Crime por km² (em vez de habitante)": montar_painel(expo=expo_km2),
    "Crime sem defasagem": montar_painel(expo=expo_hab0),
    "Trimestre (em vez de mês)": montar_painel(freq="Q"),
}
linhas = []
for nome, frame in variantes.items():
    ef = ["tempo", "cisp_tip"]
    linhas.append({"variante": nome, **estimar(frame, ef)})
for n in sorted(TIER_COLUMNS, reverse=True):
    frame = montar_painel(coluna_x=f"log_taxa_nivel{n}")
    linhas.append({"variante": f"Só nível {n}: {TIER_NAMES[n]}", **estimar(frame, ESPEC_DENTRO)})
robustez = pd.DataFrame(linhas).set_index("variante")
print("Efeito do crime no preço, com efeito fixo de CISP x tipologia e de tempo (cada linha é uma regressão):")
display(robustez[COLUNAS].round(3))

## Placebo

Troca a série de crime de cada CISP pela de outra CISP. Se o efeito real cabe na distribuição dos placebos, ele não se distingue de ruído.

In [ ]:
# Placebo: troca a série de crime de cada CISP pela de outra CISP (mesmo calendário). Se o β real estiver
# dentro da distribuição dos placebos, ele não se distingue de ruído.
n_perm = 300
sw = np.sqrt(painel["n"].to_numpy(float))
dummies = [pd.get_dummies(painel[e], drop_first=True, dtype=float).to_numpy() for e in ESPEC_DENTRO]
D = np.column_stack([np.ones(len(painel))] + dummies) * sw[:, None]
Q, _ = np.linalg.qr(D)
yt = sw * painel["y"].to_numpy()
yt = yt - Q @ (Q.T @ yt)

matriz = expo_hab["intensidade"].unstack("cisp")  # data x cisp
cisps = list(matriz.columns)
indice_data = matriz.index.get_indexer(painel["periodo"])
rng = np.random.default_rng(RANDOM_STATE)


def beta_com(coluna_cisp_por_linha):
    x = matriz.to_numpy()[indice_data, coluna_cisp_por_linha]
    ok = ~np.isnan(x)
    xw = sw * np.where(ok, x, 0)
    xt = xw - Q @ (Q.T @ xw)
    return (xt[ok] @ yt[ok]) / (xt[ok] @ xt[ok]), ok.mean()


cisp_idx = np.array([cisps.index(c) for c in painel["cisp"]])
beta_real, _ = beta_com(cisp_idx)
betas = []
for _ in range(n_perm):
    perm = rng.permutation(len(cisps))
    b, cobertura = beta_com(perm[cisp_idx])
    betas.append(b)
betas = np.array(betas)
print(f"β real = {beta_real:.4f}; placebos: média {betas.mean():.4f}, desvio-padrão {betas.std():.4f}")
print(f"p de permutação (|β placebo| >= |β real|): {(np.abs(betas) >= abs(beta_real)).mean():.3f}")

## Modelo preditor

Prevemos o ln(R$/m²) com gradient boosting, dividindo o tempo em treino (até 2020), validação (2021 e 2022) e teste (2023 em diante). A pergunta é se o crime melhora a previsão além do preço de 12 meses antes.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, r2_score

# Um registro por CISP x mês x tipologia, com o preço de 12 meses antes (só passado) e a exposição ao crime.
atras = painel[["cisp", "periodo", "tipologia", "y"]].assign(periodo=lambda d: d["periodo"] + pd.DateOffset(months=12)).rename(columns={"y": "y_lag12"})
modelo_df = painel.merge(atras, on=["cisp", "periodo", "tipologia"], how="inner", validate="one_to_one")
modelo_df["tipologia_cod"] = (modelo_df["tipologia"] == "CASA").astype(int)
modelo_df["mes_do_ano"] = modelo_df["periodo"].dt.month

BASICO = ["tipologia_cod", "mes_do_ano", "cisp"]
CRIME = ["intensidade"] + [f"log_taxa_nivel{n}" for n in sorted(TIER_COLUMNS, reverse=True)]
CONJUNTOS = {
    "Básico (tipologia, mês, CISP)": BASICO,
    "Básico + crime": BASICO + CRIME,
    "Básico + preço de 12 meses antes": BASICO + ["y_lag12"],
    "Básico + preço de 12 meses antes + crime": BASICO + ["y_lag12"] + CRIME,
}
TREINO_ATE, VALIDACAO_ATE = pd.Timestamp("2021-01-01"), pd.Timestamp("2023-01-01")
treino = modelo_df[modelo_df["periodo"] < TREINO_ATE]
validacao = modelo_df[(modelo_df["periodo"] >= TREINO_ATE) & (modelo_df["periodo"] < VALIDACAO_ATE)]
teste = modelo_df[modelo_df["periodo"] >= VALIDACAO_ATE]
for nome, d in [("treino", treino), ("validação", validacao), ("teste", teste)]:
    print(f"{nome:10s} {len(d):>6,} linhas, {d['periodo'].min():%m/%Y} a {d['periodo'].max():%m/%Y}")
assert treino["periodo"].max() < validacao["periodo"].min() <= validacao["periodo"].max() < teste["periodo"].min()


def metricas(y_log, pred_log):
    y, p = np.exp(y_log), np.exp(pred_log)
    return {"r2_log": r2_score(y_log, pred_log), "mae_reais_m2": mean_absolute_error(y, p),
            "erro_mediano_pct": float(np.median(np.abs(p - y) / y) * 100)}


def ajustar(cols, dados, params):
    categoricas = [c for c in cols if c == "cisp"]
    modelo = HistGradientBoostingRegressor(
        max_iter=200, early_stopping=False, random_state=RANDOM_STATE, categorical_features=categoricas or None, **params
    )
    return modelo.fit(dados[cols], dados["y"])


GRADE = [{"max_depth": d, "learning_rate": lr, "min_samples_leaf": leaf} for d in (3, 6) for lr in (0.05, 0.1) for leaf in (20, 100)]

In [ ]:
# Hiperparâmetros escolhidos na VALIDAÇÃO (2021-2022); o TESTE (2023 em diante) é usado uma vez, com o modelo refeito em treino + validação.
melhores = {}
linhas = []
fit_final = pd.concat([treino, validacao])
for nome, cols in CONJUNTOS.items():
    ranking = [(r2_score(validacao["y"], ajustar(cols, treino, p).predict(validacao[cols])), p) for p in GRADE]
    r2_val, melhor = max(ranking, key=lambda t: t[0])
    melhores[nome] = melhor
    m = metricas(teste["y"].to_numpy(), ajustar(cols, fit_final, melhor).predict(teste[cols]))
    linhas.append({"conjunto": nome, "r2_validacao": r2_val, **{f"teste_{k}": v for k, v in m.items()}})
ingenuo = metricas(teste["y"].to_numpy(), teste["y_lag12"].to_numpy())
linhas.insert(0, {"conjunto": "Sem modelo (preço de 12 meses antes)", "r2_validacao": np.nan, **{f"teste_{k}": v for k, v in ingenuo.items()}})
resultado_teste = pd.DataFrame(linhas).set_index("conjunto")
display(resultado_teste.round(3))

# Origens móveis: para cada ano de teste, treina só com o passado e usa os hiperparâmetros acima.
registros = []
for ano in (2021, 2022, 2023, 2024, 2025):
    ini, fim = pd.Timestamp(f"{ano}-01-01"), pd.Timestamp(f"{ano + 1}-01-01")
    tr, te = modelo_df[modelo_df["periodo"] < ini], modelo_df[(modelo_df["periodo"] >= ini) & (modelo_df["periodo"] < fim)]
    if len(te) == 0:
        continue
    registros.append({"ano_teste": ano, "conjunto": "Sem modelo (preço de 12 meses antes)", **metricas(te["y"].to_numpy(), te["y_lag12"].to_numpy())})
    for nome, cols in CONJUNTOS.items():
        registros.append({"ano_teste": ano, "conjunto": nome, **metricas(te["y"].to_numpy(), ajustar(cols, tr, melhores[nome]).predict(te[cols]))})
origens = pd.DataFrame(registros)
print("R² no teste de cada ano (treino só com os anos anteriores):")
display(origens.pivot(index="conjunto", columns="ano_teste", values="r2_log").round(3))
print("Erro mediano % de cada ano:")
display(origens.pivot(index="conjunto", columns="ano_teste", values="erro_mediano_pct").round(1))

In [ ]:
cols = CONJUNTOS["Básico + preço de 12 meses antes + crime"]
modelo_final = ajustar(cols, fit_final, melhores["Básico + preço de 12 meses antes + crime"])
imp = permutation_importance(modelo_final, teste[cols], teste["y"], n_repeats=5, random_state=RANDOM_STATE, scoring="r2")
importancia = pd.Series(imp.importances_mean, index=cols).sort_values(ascending=False)
print("Queda de R² no teste ao embaralhar cada variável:")
display(importancia.round(4).to_frame("queda_r2"))
print(f"Soma das variáveis de crime: {importancia[CRIME].sum():.4f}")

## Leitura dos resultados

O painel tem 3.923 células CISP × mês × tipologia, em 38 CISPs, de 04/2014 a 07/2026. Dentro de cada CISP o crime varia mais que no Airbnb (desvio-padrão de 0,23 contra 0,15), e o efeito mínimo detectável cai para 5,0% do preço por desvio-padrão de crime (2,5% com tendência por CISP), contra cerca de 8% no Airbnb.

No período todo, com efeito fixo de CISP, o crime tem associação positiva no limite da significância: +3,8% no R$/m² por desvio-padrão (IC −0,4% a +7,7%, p = 0,07). O nível 1, furtos e estelionato, chega a +5,2% (p = 0,011), e o placebo dá p = 0,027. Esse sinal some quando se isola o curto prazo: antes e depois de 2022 separados o efeito é de +0,2% e +0,3% (p = 0,90 e 0,88), e ao tirar a tendência linear de cada CISP ele vai para −0,4% (IC −2,2% a +1,3%, p = 0,69). A associação do período todo vem de mudanças lentas dos bairros, não de oscilações do crime de um mês para outro, e esse desenho não consegue separar uma coisa da outra.

O crime não melhora a previsão além do preço de 12 meses antes: R² de 0,923 sem crime e 0,921 com crime no teste de 2023 em diante, e a diferença por ano de teste fica entre −0,008 e +0,001. As variáveis de crime somam 0,017 de queda de R² na importância por permutação, contra 0,889 do preço defasado.